<a href="https://colab.research.google.com/github/anamitra-tech/ML-Projects/blob/main/EmotionTracker4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.8 MB/s eta 0:00:00


In [6]:
# Arvyax Reflective Session - Dual Output Neural Network
# Built from scratch using numpy only (no torch/tf dependency)
#
# What this does:
#   - Predicts emotional state (6 classes) from journal text + ambience + face emotion etc.
#   - Predicts intensity (1-5 regression) from numeric session data
#   - Generates a recommendation using a soft attention scoring mechanism
#
# Note: the attention layer, early stopping, and LR scheduler are all manual
# since we're not using any deep learning framework here.

import numpy as np
import pandas as pd
import re
import json
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_squared_error, r2_score

print("=" * 60)
print("Arvyax - Dual Output NN Pipeline")
print("=" * 60)


# ==============================================================
# DATA LOADING
# ==============================================================

train_df = pd.read_csv('/content/Sample_arvyax_reflective_dataset.xlsx - Dataset_120.csv')
test_df  = pd.read_csv('/content/arvyax_test_inputs_120.xlsx - Sheet1.csv')

print(f"\nTraining samples : {len(train_df)}")
print(f"Test samples     : {len(test_df)}")
print(f"Columns          : {list(train_df.columns)}")


# ==============================================================
# TEXT FEATURE EXTRACTION
# proximity-weighted emotion scoring per journal entry
# if the ambience word shows up in the journal text, keywords
# near it get a higher weight (simulating attention over tokens)
# ==============================================================

AMBIENCE_TYPES = ["ocean", "forest", "mountain", "rain", "cafe"]

# vocab clusters for each emotion - built by hand after reading the dataset
EMOTION_VOCAB = {
    "calm": ["calm", "settle", "settled", "quiet", "peaceful", "lighter", "ease", "grounded",
             "slow", "soft", "slowed", "soften", "serene", "centered"],
    "restless": ["restless", "jumpy", "racing", "fidgety", "scattered", "distracted", "buzz",
                 "switch", "bounce", "itchy", "unable", "wander"],
    "focused": ["focus", "focused", "clear", "plan", "organize", "prioritize", "lock",
                "concentrate", "ready", "tackle", "step", "start"],
    "overwhelmed": ["overwhelmed", "overloaded", "heavy", "pressure", "carrying", "flooded",
                    "piled", "drained", "everything", "too much", "behind", "hard"],
    "neutral": ["normal", "same", "steady", "average", "fine", "okay", "nothing", "fairly",
                "not much", "just", "neutral", "aware"],
    "mixed": ["mixed", "split", "between", "both", "part", "two", "comforted", "distracted",
              "better and not", "uneasy", "lingering", "also"]
}


def ambience_weighted_text_vector(journal, ambience):
    # returns a 7-dim vector: 6 emotion scores (normalized) + 1 ambience-in-text flag
    if not isinstance(journal, str):
        journal = ""
    if not isinstance(ambience, str):
        ambience = ""

    jl = journal.lower()
    al = ambience.lower()

    ambience_in_text = float(al in jl)

    tokens = re.findall(r'\b\w+\b', jl)
    ambience_positions = [i for i, t in enumerate(tokens) if t == al]

    emotion_scores = []
    for emo, keywords in EMOTION_VOCAB.items():
        score = 0.0
        for kw in keywords:
            if kw in jl:
                base = 1.0
                # boost keywords that appear close to the ambience word in the text
                if ambience_positions:
                    kw_positions = [i for i, t in enumerate(tokens) if t == kw.split()[0]]
                    for ap in ambience_positions:
                        for kp in kw_positions:
                            dist = abs(ap - kp)
                            proximity_weight = 2.0 / (1.0 + dist * 0.1)
                            base = max(base, proximity_weight)
                score += base
        emotion_scores.append(score)

    total = sum(emotion_scores) + 1e-9
    emotion_scores = [s / total for s in emotion_scores]

    return np.array(emotion_scores + [ambience_in_text], dtype=np.float32)


def get_text_features(journal, ambience):
    return ambience_weighted_text_vector(journal, ambience)


# ==============================================================
# FACE EMOTION x PREVIOUS DAY MOOD INTERACTION
# instead of hardcoding rules like calm+tired_face = X,
# we encode both as one-hot and let the network learn the
# interaction weights during training
# ==============================================================

FACE_EMOTIONS = ["calm_face", "happy_face", "neutral_face", "tired_face", "tense_face", "none", ""]
PREV_MOODS    = ["calm", "focused", "mixed", "neutral", "overwhelmed", "restless", "", None]


def encode_face_mood_interaction(face, prev_mood):
    # 7-dim one-hot for face + 8-dim one-hot for prev_mood = 15 dims total
    face = str(face).strip().lower() if pd.notna(face) else "none"
    prev = str(prev_mood).strip().lower() if pd.notna(prev_mood) else ""

    f_vec = np.zeros(len(FACE_EMOTIONS), dtype=np.float32)
    for i, fe in enumerate(FACE_EMOTIONS):
        if fe == face:
            f_vec[i] = 1.0
            break

    p_vec = np.zeros(len(PREV_MOODS), dtype=np.float32)
    for i, pm in enumerate(PREV_MOODS):
        if str(pm).lower() == prev:
            p_vec[i] = 1.0
            break

    return np.concatenate([f_vec, p_vec])


def encode_reflection_quality(rq):
    # treating this as ordinal: vague < conflicted < clear
    mapping = {"vague": 0.0, "conflicted": 0.5, "clear": 1.0}
    val = mapping.get(str(rq).lower().strip(), 0.25)
    return np.array([val], dtype=np.float32)


# ==============================================================
# PREVIOUS DAY SEMANTIC SIMILARITY
# simple lexical overlap between journal text and each mood's
# keyword cluster - gives a 6-dim vector per sample
# ==============================================================

MOOD_VOCAB_MAP = {mood: set(kws) for mood, kws in EMOTION_VOCAB.items()}


def previous_day_semantic_sim(journal, prev_mood):
    if not isinstance(journal, str):
        journal = ""
    tokens = set(re.findall(r'\b\w+\b', journal.lower()))
    sims = []
    for mood, vocab in MOOD_VOCAB_MAP.items():
        overlap = len(tokens & vocab)
        sim = overlap / (np.sqrt(len(tokens) + 1) * np.sqrt(len(vocab) + 1))
        sims.append(sim)
    return np.array(sims, dtype=np.float32)


# ==============================================================
# FEATURE ASSEMBLY
# Classification features (for Y1 - emotional state):
#   text+ambience vector   -> 7 dims
#   face x prev_mood       -> 15 dims
#   semantic similarity    -> 6 dims
#   reflection quality     -> 1 dim
#   total                  -> 29 dims
# ==============================================================

def build_cls_features(df):
    rows = []
    for _, row in df.iterrows():
        t = get_text_features(row.get("journal_text", ""), row.get("ambience_type", ""))
        i = encode_face_mood_interaction(row.get("face_emotion_hint", ""), row.get("previous_day_mood", ""))
        s = previous_day_semantic_sim(row.get("journal_text", ""), row.get("previous_day_mood", ""))
        r = encode_reflection_quality(row.get("reflection_quality", "vague"))
        rows.append(np.concatenate([t, i, s, r]))
    return np.array(rows, dtype=np.float32)


# Regression features (for Y2 - intensity):
# using all numeric session columns
NUMERIC_COLS = ["duration_min", "sleep_hours", "energy_level", "stress_level"]


def build_reg_features(df):
    out = []
    for col in NUMERIC_COLS:
        vals = pd.to_numeric(df[col], errors="coerce")
        vals = vals.fillna(vals.median())
        out.append(vals.values.reshape(-1, 1))
    return np.hstack(out).astype(np.float32)


# ==============================================================
# LABEL ENCODING
# ==============================================================

EMOTIONAL_STATES = ["calm", "focused", "mixed", "neutral", "overwhelmed", "restless"]


def encode_labels(df):
    le = LabelEncoder()
    le.classes_ = np.array(EMOTIONAL_STATES)
    y = le.transform(df["emotional_state"].str.lower().str.strip())
    return y, le


# ==============================================================
# NEURAL NETWORK - PURE NUMPY
# activation functions
# ==============================================================

def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(float)

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def cross_entropy_loss(pred, y_onehot):
    return -np.mean(np.sum(y_onehot * np.log(pred + 1e-9), axis=1))

def mse_loss(pred, y):
    return np.mean((pred.flatten() - y) ** 2)


class AttentionLayer:
    # single-head attention
    # each input feature set gets a learned query/key/value projection
    # the score tells us which features matter most for prediction
    def __init__(self, d_model, d_k=8):
        scale = np.sqrt(2.0 / d_model)
        self.Wq = np.random.randn(d_model, d_k) * scale
        self.Wk = np.random.randn(d_model, d_k) * scale
        self.Wv = np.random.randn(d_model, d_k) * scale
        self.d_k = d_k
        self.cache = {}

    def forward(self, X):
        Q = X @ self.Wq
        K = X @ self.Wk
        V = X @ self.Wv
        scale = np.sqrt(self.d_k)
        scores = (Q * K).sum(axis=1, keepdims=True) / scale
        alpha = sigmoid(scores)
        out = alpha * V
        self.cache = {"X": X, "Q": Q, "K": K, "V": V, "alpha": alpha}
        return out, alpha

    def backward(self, dout, lr):
        alpha = self.cache["alpha"]
        V = self.cache["V"]
        X = self.cache["X"]
        Q = self.cache["Q"]
        K = self.cache["K"]

        dV = alpha * dout
        dalpha = (dout * V).sum(axis=1, keepdims=True)
        dscores = dalpha * alpha * (1 - alpha)

        scale = np.sqrt(self.d_k)
        dQ = dscores * K / scale
        dK = dscores * Q / scale

        self.Wq -= lr * (X.T @ dQ)
        self.Wk -= lr * (X.T @ dK)
        self.Wv -= lr * (X.T @ dV)

        return dout


class DualOutputNN:
    # Architecture:
    #   cls input (29) -> attention (29->8) -> concat (37) -> FC(64) -> FC(32) -> softmax(6)
    #   reg input (4)  -> FC(32) -> FC(1) linear
    #
    # two separate loss functions, both backpropagated
    # classification loss weighted at 1.0, regression at 0.5

    def __init__(self, input_dim, hidden1=64, hidden2=32, n_classes=6,
                 reg_input_dim=4, reg_hidden=32):
        self.input_dim = input_dim
        self.dk = 8

        self.attn = AttentionLayer(input_dim, d_k=self.dk)

        s2 = np.sqrt(2.0 / (input_dim + self.dk))
        s3 = np.sqrt(2.0 / hidden1)
        s4 = np.sqrt(2.0 / hidden2)

        self.W1 = np.random.randn(input_dim + self.dk, hidden1) * s2
        self.b1 = np.zeros((1, hidden1))
        self.W2 = np.random.randn(hidden1, hidden2) * s3
        self.b2 = np.zeros((1, hidden2))
        self.Wc = np.random.randn(hidden2, n_classes) * s4
        self.bc = np.zeros((1, n_classes))

        sr = np.sqrt(2.0 / reg_input_dim)
        self.Wr1 = np.random.randn(reg_input_dim, reg_hidden) * sr
        self.br1 = np.zeros((1, reg_hidden))
        self.Wr2 = np.random.randn(reg_hidden, 1) * np.sqrt(2.0 / reg_hidden)
        self.br2 = np.zeros((1, 1))

        self.cache = {}

    def forward(self, X_cls, X_reg):
        attn_out, alpha = self.attn.forward(X_cls)
        X_aug = np.concatenate([X_cls, attn_out], axis=1)

        z1 = X_aug @ self.W1 + self.b1
        a1 = relu(z1)
        z2 = a1 @ self.W2 + self.b2
        a2 = relu(z2)

        zc = a2 @ self.Wc + self.bc
        cls_pred = softmax(zc)

        zr1 = X_reg @ self.Wr1 + self.br1
        ar1 = relu(zr1)
        zr2 = ar1 @ self.Wr2 + self.br2

        self.cache = {
            "X_aug": X_aug, "X_reg": X_reg,
            "z1": z1, "a1": a1, "z2": z2, "a2": a2,
            "cls_pred": cls_pred,
            "zr1": zr1, "ar1": ar1, "reg_pred": zr2,
        }
        return cls_pred, zr2

    def backward(self, y_onehot, y_reg, lr):
        B = y_onehot.shape[0]
        c = self.cache

        # classification head gradients
        d_zc = (c["cls_pred"] - y_onehot) / B
        dWc  = c["a2"].T @ d_zc
        dbc  = d_zc.sum(axis=0, keepdims=True)
        d_a2 = d_zc @ self.Wc.T

        # regression head gradients
        d_zr2 = 2 * (c["reg_pred"] - y_reg.reshape(-1, 1)) / B
        dWr2  = c["ar1"].T @ d_zr2
        dbr2  = d_zr2.sum(axis=0, keepdims=True)
        d_ar1 = d_zr2 @ self.Wr2.T
        d_zr1 = d_ar1 * relu_grad(c["zr1"])
        dWr1  = c["X_reg"].T @ d_zr1
        dbr1  = d_zr1.sum(axis=0, keepdims=True)

        # shared encoder gradients (flowing from cls head)
        d_z2 = d_a2 * relu_grad(c["z2"])
        dW2  = c["a1"].T @ d_z2
        db2  = d_z2.sum(axis=0, keepdims=True)
        d_a1 = d_z2 @ self.W2.T
        d_z1 = d_a1 * relu_grad(c["z1"])
        dW1  = c["X_aug"].T @ d_z1
        db1  = d_z1.sum(axis=0, keepdims=True)

        # pass gradient back to attention (only the attn_out slice)
        d_aug = d_z1 @ self.W1.T
        self.attn.backward(d_aug[:, self.input_dim:], lr)

        # weight updates
        self.W1  -= lr * dW1;  self.b1  -= lr * db1
        self.W2  -= lr * dW2;  self.b2  -= lr * db2
        self.Wc  -= lr * dWc;  self.bc  -= lr * dbc
        self.Wr1 -= lr * dWr1; self.br1 -= lr * dbr1
        self.Wr2 -= lr * dWr2; self.br2 -= lr * dbr2


# ==============================================================
# EARLY STOPPING + LR SCHEDULER (manual, no framework needed)
# ==============================================================

class EarlyStopping:
    # stops training when val loss stops improving
    # patience = how many epochs to wait before stopping
    def __init__(self, patience=25, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = np.inf
        self.counter    = 0
        self.best_epoch = 0
        self.stop       = False
        self.best_weights = None

    def check(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_epoch = epoch
            # snapshot current weights
            self.best_weights = {
                "W1": model.W1.copy(), "b1": model.b1.copy(),
                "W2": model.W2.copy(), "b2": model.b2.copy(),
                "Wc": model.Wc.copy(), "bc": model.bc.copy(),
                "Wr1": model.Wr1.copy(), "br1": model.br1.copy(),
                "Wr2": model.Wr2.copy(), "br2": model.br2.copy(),
                "attn_Wq": model.attn.Wq.copy(),
                "attn_Wk": model.attn.Wk.copy(),
                "attn_Wv": model.attn.Wv.copy(),
            }
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def restore_best(self, model):
        if self.best_weights is not None:
            model.W1      = self.best_weights["W1"]
            model.b1      = self.best_weights["b1"]
            model.W2      = self.best_weights["W2"]
            model.b2      = self.best_weights["b2"]
            model.Wc      = self.best_weights["Wc"]
            model.bc      = self.best_weights["bc"]
            model.Wr1     = self.best_weights["Wr1"]
            model.br1     = self.best_weights["br1"]
            model.Wr2     = self.best_weights["Wr2"]
            model.br2     = self.best_weights["br2"]
            model.attn.Wq = self.best_weights["attn_Wq"]
            model.attn.Wk = self.best_weights["attn_Wk"]
            model.attn.Wv = self.best_weights["attn_Wv"]
            print(f"  restored best weights from epoch {self.best_epoch + 1}")


class ReduceLROnPlateau:
    # reduces learning rate when val loss plateaus
    # same idea as pytorch's ReduceLROnPlateau but written manually
    def __init__(self, initial_lr, factor=0.5, patience=15, min_lr=1e-5, min_delta=1e-4):
        self.lr        = initial_lr
        self.factor    = factor
        self.patience  = patience
        self.min_lr    = min_lr
        self.min_delta = min_delta
        self.best_loss = np.inf
        self.counter   = 0

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                new_lr = max(self.lr * self.factor, self.min_lr)
                if new_lr < self.lr:
                    print(f"  LR reduced: {self.lr:.6f} -> {new_lr:.6f}")
                    self.lr = new_lr
                self.counter = 0
        return self.lr


# ==============================================================
# BUILD FEATURES
# ==============================================================

print("\n[1/4] Building features...")

X_cls_raw = build_cls_features(train_df)
X_reg_raw = build_reg_features(train_df)

cls_scaler = StandardScaler()
X_cls = cls_scaler.fit_transform(X_cls_raw)

reg_scaler = StandardScaler()
X_reg = reg_scaler.fit_transform(X_reg_raw)

y_cls, label_enc = encode_labels(train_df)
y_reg = train_df["intensity"].astype(float).values

n_classes = len(EMOTIONAL_STATES)
y_onehot  = np.eye(n_classes)[y_cls]

input_dim = X_cls.shape[1]
print(f"  cls feature dim : {X_cls.shape}")
print(f"  reg feature dim : {X_reg.shape}")

(X_cls_tr, X_cls_val,
 X_reg_tr, X_reg_val,
 y_oh_tr, y_oh_val,
 y_reg_tr, y_reg_val) = train_test_split(
    X_cls, X_reg, y_onehot, y_reg,
    test_size=0.15, random_state=42, stratify=y_cls
)

print(f"  train / val split : {X_cls_tr.shape[0]} / {X_cls_val.shape[0]}")


# ==============================================================
# TRAINING LOOP
# ==============================================================

print("\n[2/4] Training...")

N_EPOCHS   = 500   # max epochs - early stopping usually kicks in before this
BATCH_SIZE = 32
INIT_LR    = 0.005
LAMBDA_CLS = 1.0
LAMBDA_REG = 0.5

model = DualOutputNN(
    input_dim=input_dim, hidden1=64, hidden2=32,
    n_classes=6, reg_input_dim=4, reg_hidden=32
)

early_stop = EarlyStopping(patience=25, min_delta=1e-4)
lr_sched   = ReduceLROnPlateau(initial_lr=INIT_LR, factor=0.5, patience=15, min_lr=1e-5)

train_losses = []
val_losses   = []
lr_history   = []

for epoch in range(N_EPOCHS):
    idx        = np.random.permutation(X_cls_tr.shape[0])
    epoch_loss = 0.0
    current_lr = lr_sched.lr

    for start in range(0, len(idx), BATCH_SIZE):
        b    = idx[start:start + BATCH_SIZE]
        Xc_b = X_cls_tr[b];  Xr_b = X_reg_tr[b]
        yc_b = y_oh_tr[b];   yr_b = y_reg_tr[b]

        cls_p, reg_p = model.forward(Xc_b, Xr_b)
        loss = (LAMBDA_CLS * cross_entropy_loss(cls_p, yc_b)
                + LAMBDA_REG * mse_loss(reg_p, yr_b))
        epoch_loss += loss
        model.backward(yc_b, yr_b, lr=current_lr)

    avg_train_loss = epoch_loss / max(1, len(idx) // BATCH_SIZE)

    # validation pass
    cls_vp, reg_vp = model.forward(X_cls_val, X_reg_val)
    val_loss = (LAMBDA_CLS * cross_entropy_loss(cls_vp, y_oh_val)
                + LAMBDA_REG * mse_loss(reg_vp, y_reg_val))

    train_losses.append(float(avg_train_loss))
    val_losses.append(float(val_loss))
    lr_history.append(current_lr)

    lr_sched.step(val_loss)
    early_stop.check(val_loss, model, epoch)

    if (epoch + 1) % 50 == 0:
        val_acc = np.mean(np.argmax(cls_vp, axis=1) == np.argmax(y_oh_val, axis=1))
        val_mse = mse_loss(reg_vp, y_reg_val)
        print(f"  epoch {epoch+1:3d} | "
              f"train loss: {avg_train_loss:.4f} | "
              f"val loss: {val_loss:.4f} | "
              f"val acc: {val_acc:.3f} | "
              f"val mse: {val_mse:.4f} | "
              f"lr: {current_lr:.5f}")

    if early_stop.stop:
        print(f"\n  early stopping triggered at epoch {epoch + 1}")
        break

# restore best checkpoint
early_stop.restore_best(model)
print(f"  best val loss: {early_stop.best_loss:.4f} at epoch {early_stop.best_epoch + 1}")


# ==============================================================
# EVALUATION ON FULL TRAINING SET
# ==============================================================

print("\n[3/4] Evaluating on training set...")

cls_pred_all, reg_pred_all = model.forward(X_cls, X_reg)
y_pred_cls    = np.argmax(cls_pred_all, axis=1)
y_pred_labels = label_enc.classes_[y_pred_cls]
y_true_labels = label_enc.classes_[y_cls]

print("\nClassification Report - Emotional State")
print(classification_report(y_true_labels, y_pred_labels, zero_division=0))

r2   = r2_score(y_reg, reg_pred_all.flatten())
rmse = np.sqrt(mean_squared_error(y_reg, reg_pred_all.flatten()))
print(f"Regression - Intensity")
print(f"  R2   : {r2:.4f}")
print(f"  RMSE : {rmse:.4f}")


# ==============================================================
# ATTENTION-BASED RECOMMENDATION ENGINE
#
# how it works:
#   1. query Q = predicted class probs (6) + normalized intensity (1) -> shape (7,)
#   2. key matrix K = 8 template embeddings, each aligned to the emotion space
#   3. score = softmax(K @ Q / sqrt(7))
#   4. pick the template with the highest attention score
#
# completely soft selection, no if/else anywhere
# ==============================================================

RECOMMENDATION_TEMPLATES = [
    {
        "label": "deep_work",
        "keywords": ["focused", "calm", "organized", "clear"],
        "template": lambda amb, tod, dur, sl: (
            f"Your mind is in a receptive state right now. Use this window for deep work or planning. "
            f"The {amb} ambience supported your concentration today - consider using it again next time. "
            f"Start with the most demanding task while this clarity holds."
        )
    },
    {
        "label": "gentle_reset",
        "keywords": ["calm", "settled", "lighter", "peaceful"],
        "template": lambda amb, tod, dur, sl: (
            f"You've settled into a quieter headspace. The {amb} ambience helped anchor this. "
            f"A short mindful pause or light movement will carry this feeling further into {tod}."
        )
    },
    {
        "label": "grounding_practice",
        "keywords": ["restless", "jumpy", "scattered", "racing"],
        "template": lambda amb, tod, dur, sl: (
            f"Your system is still running fast. The {amb} sounds can work as a grounding anchor - "
            f"try syncing your breath slowly to the ambient rhythm. Stick to one task at a time to bring the buzz down."
        )
    },
    {
        "label": "emotional_offload",
        "keywords": ["overwhelmed", "heavy", "flooded", "pressure"],
        "template": lambda amb, tod, dur, sl: (
            f"You're carrying a lot right now. The {amb} ambience has been doing its quiet work. "
            f"Before returning to demands, try a short journal dump or a walk to release some of the load."
        )
    },
    {
        "label": "dual_awareness",
        "keywords": ["mixed", "split", "between", "uneasy"],
        "template": lambda amb, tod, dur, sl: (
            f"Two emotional currents are running at the same time. The {amb} setting helped soften the gap between them. "
            f"Don't force a resolution - pick one anchor task and let it pull you gently forward."
        )
    },
    {
        "label": "steady_continuity",
        "keywords": ["neutral", "steady", "same", "fine"],
        "template": lambda amb, tod, dur, sl: (
            f"Your baseline is stable today. The {amb} session kept things even. "
            f"This is a solid state for routine work or small creative steps during {tod}."
        )
    },
    {
        "label": "rest_recovery",
        "keywords": ["tired", "tired_face", "drained", "exhausted"],
        "template": lambda amb, tod, dur, sl: (
            f"Fatigue is showing up clearly. The {amb} soundscape gave you some softening but recovery needs more. "
            f"Sleep and genuine stillness are the highest-return action right now."
        )
    },
    {
        "label": "high_intensity_redirect",
        "keywords": ["tense_face", "tense", "wound", "unable"],
        "template": lambda amb, tod, dur, sl: (
            f"Physical tension is elevated. Use the {amb} ambience as a reset point between tasks. "
            f"Break larger obligations into smaller concrete steps to reduce the felt pressure."
        )
    },
]

KEY_DIM = len(EMOTIONAL_STATES) + 1  # 7 dims for the query/key space


def build_template_key(keywords):
    key = np.zeros(KEY_DIM, dtype=np.float32)
    state_map = {s: i for i, s in enumerate(EMOTIONAL_STATES)}
    for kw in keywords:
        for state, sidx in state_map.items():
            if kw in state or state in kw or kw in MOOD_VOCAB_MAP.get(state, set()):
                key[sidx] += 1.0
        if kw in ["tired", "tense", "tense_face", "exhausted", "drained"]:
            key[-1] += 1.0
    key /= (np.linalg.norm(key) + 1e-9)
    return key


KEY_MATRIX = np.array([build_template_key(t["keywords"]) for t in RECOMMENDATION_TEMPLATES])


def attention_recommend(cls_probs, reg_pred, ambience, time_of_day, duration_min, sleep_hours):
    # build query from predicted probabilities
    intensity_norm = np.clip(reg_pred / 5.0, 0, 1)
    Q = np.append(cls_probs, intensity_norm).astype(np.float32)

    # score each template against the query
    scores  = (KEY_MATRIX @ Q) / np.sqrt(KEY_DIM)
    attn_w  = np.exp(scores - scores.max())
    attn_w /= attn_w.sum()

    top_idx      = int(np.argmax(attn_w))
    top_template = RECOMMENDATION_TEMPLATES[top_idx]

    # sleep: if under 6h suggest 8h
    sleep_rec = 8 if sleep_hours < 6 else round(sleep_hours, 1)

    rec_text = top_template["template"](ambience, time_of_day, duration_min, sleep_hours)

    return {
        "recommendation":          rec_text,
        "attention_weights":       {t["label"]: float(w) for t, w in zip(RECOMMENDATION_TEMPLATES, attn_w)},
        "top_template":            top_template["label"],
        "duration_min":            int(duration_min),
        "sleep_hours_recommended": sleep_rec,
        "time_of_day":             time_of_day,
    }


# ==============================================================
# INFERENCE ON TEST DATA
# test_df was never seen during training - loaded separately at top
# ==============================================================

print("\n[4/4] Running on test data...")

X_cls_test = cls_scaler.transform(build_cls_features(test_df))
X_reg_test = reg_scaler.transform(build_reg_features(test_df))

cls_pred_test, reg_pred_test = model.forward(X_cls_test, X_reg_test)

y_pred_cls_test    = np.argmax(cls_pred_test, axis=1)
y_pred_labels_test = label_enc.classes_[y_pred_cls_test]
y_pred_reg_test    = np.clip(reg_pred_test.flatten(), 1, 5)

results = []
for i, row in test_df.iterrows():
    idx = i - test_df.index[0]
    amb = str(row.get("ambience_type", "")).lower()
    tod = str(row.get("time_of_day", "")).lower()
    dur = float(row.get("duration_min", 10))
    sl  = float(row.get("sleep_hours", 7)) if pd.notna(row.get("sleep_hours")) else 7.0

    rec = attention_recommend(
        cls_pred_test[idx], y_pred_reg_test[idx], amb, tod, dur, sl
    )

    results.append({
        "id":                        row["id"],
        "predicted_emotional_state": y_pred_labels_test[idx],
        "predicted_intensity":       round(float(y_pred_reg_test[idx]), 2),
        "recommendation":            rec["recommendation"],
        "top_template":              rec["top_template"],
        "duration_min":              rec["duration_min"],
        "sleep_hours_recommended":   rec["sleep_hours_recommended"],
        "time_of_day":               rec["time_of_day"],
        "attention_weights":         rec["attention_weights"],
        "cls_confidence":            round(float(cls_pred_test[idx].max()), 3),
        "ambience_type":             row.get("ambience_type", ""),
    })

out_df = pd.DataFrame(results)

print(f"\n  predictions done for {len(out_df)} test samples")
print(f"\n  emotional state breakdown:")
print(out_df["predicted_emotional_state"].value_counts().to_string())
print(f"\n  intensity stats:")
print(out_df["predicted_intensity"].describe().round(3).to_string())

out_df.to_csv("arvyax_predictions.csv", index=False)
print("\ndone. predictions saved to arvyax_predictions.csv")
print("=" * 60)


Arvyax - Dual Output NN Pipeline

Training samples : 1200
Test samples     : 120
Columns          : ['id', 'journal_text', 'ambience_type', 'duration_min', 'sleep_hours', 'energy_level', 'stress_level', 'time_of_day', 'previous_day_mood', 'face_emotion_hint', 'reflection_quality', 'emotional_state', 'intensity']

[1/4] Building features...
  cls feature dim : (1200, 29)
  reg feature dim : (1200, 4)
  train / val split : 1020 / 180

[2/4] Training...
  epoch  50 | train loss: 2.3775 | val loss: 2.6105 | val acc: 0.400 | val mse: 2.0029 | lr: 0.00500
  LR reduced: 0.005000 -> 0.002500

  early stopping triggered at epoch 96
  restored best weights from epoch 71
  best val loss: 2.5908 at epoch 71

[3/4] Evaluating on training set...

Classification Report - Emotional State
              precision    recall  f1-score   support

        calm       0.48      0.62      0.54       216
     focused       0.60      0.55      0.57       193
       mixed       0.55      0.40      0.47       191
